In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("28-spark-ui-debugging")
dfs = register_views(spark)
emp = dfs["employees"]
sal = dfs["salary_history"]
dept = dfs["departments"]

## Section 1 — Generating diverse Spark UI content

In [ ]:
print("=" * 70)
print("SECTION 1: Goal — populate every Spark UI tab")
print("=" * 70)
# This script deliberately exercises different patterns so each Spark UI tab
# shows interesting content. Browse http://localhost:4040 after each section.
print("Browse http://localhost:4040 as each section runs.")

## Section 2 — Jobs tab: trigger multiple distinct jobs

In [ ]:
print("\n" + "=" * 70)
print("SECTION 2: Jobs Tab — multiple distinct jobs")
print("=" * 70)

import time
for label, action in [
    ("emp.count",   lambda: emp.count()),
    ("sal.count",   lambda: sal.count()),
    ("dept.show",   lambda: dept.show(truncate=False)),
    ("emp.groupBy", lambda: emp.groupBy("dept_id").agg(F.avg("salary")).show()),
]:
    t = time.time()
    action()
    print(f"Job '{label}': {time.time() - t:.3f}s — check Jobs tab")

## Section 3 — SQL tab: trigger SQL operations

In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: SQL Tab — SQL/DataFrame operations")
print("=" * 70)

# Each .explain() or action on a DF registered as a view shows up in SQL tab.
print("\n[SQL 1] groupBy with AVG:")
spark.sql("SELECT dept_id, COUNT(*) cnt, AVG(salary) avg_sal FROM employees GROUP BY dept_id").show()

print("\n[SQL 2] join employees + departments:")
spark.sql("SELECT e.emp_id, e.first_name, d.dept_name FROM employees e JOIN departments d USING(dept_id)").show(5)

## Section 4 — Storage tab: cache a DataFrame and inspect

In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: Storage Tab — cache emp+dept and inspect")
print("=" * 70)

emp_cached = emp.join(dept, "dept_id", "left").cache()
emp_cached.count()  # materialise the cache
print("emp+dept cached — check Storage tab at http://localhost:4040/storage")
print(f"emp_cached.is_cached: {emp_cached.is_cached}")
# Storage tab shows: fraction cached, memory used, deserialized size.

## Section 5 — Stages tab: trigger a shuffle to create multiple stages

In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Stages Tab — multi-stage job via join + groupBy")
print("=" * 70)

spark.conf.set("spark.sql.shuffle.partitions", "4")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
emp.join(sal, "emp_id").groupBy("dept_id").agg(F.sum("salary_after")).show()
print("Multi-stage job done — check Stages tab for Exchange nodes and task counts.")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

## Section 6 — Environment tab: verify key configs

In [ ]:
print("\n" + "=" * 70)
print("SECTION 6: Environment Tab — key config values")
print("=" * 70)

key_configs = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.driver.memory",
    "spark.sql.adaptive.skewJoin.enabled",
]
print("\nKey configs (also visible in Environment tab):")
for k in key_configs:
    try:
        print(f"  {k} = {spark.conf.get(k)}")
    except Exception:
        print(f"  {k} = (not set)")

## Section 7 — Detecting skew in Stages tab

In [ ]:
print("\n" + "=" * 70)
print("SECTION 7: Skew Detection in Stages Tab")
print("=" * 70)

# Create artificial skew: repartition by dept_id
# Engineering (dept_id=1) has 14/41 employees — the biggest partition.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
emp.repartition(8, "dept_id").join(sal, "emp_id").groupBy("dept_id").count().show()
print("Skew demo done — in Stages tab, look for 'Max Duration' much > 'Median Duration'.")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

## Section 8 — GC pressure (comment + advice)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 8: GC Pressure — signs and fixes (comments)")
print("=" * 70)

# GC shows up in Stages > Task Metrics > GC Time.
# Signs of GC pressure:  GC Time > 10% of Task Duration.
# Causes:
#   - executor heap too small for working set
#   - Python UDFs holding object references across rows
#   - forgetting to unpersist() after caching
# Fixes:
#   - increase executor memory (--executor-memory or spark.executor.memory)
#   - use StorageLevel.MEMORY_AND_DISK instead of MEMORY_ONLY
#   - switch Python UDFs to pandas UDFs (vectorised, much less per-row overhead)
print("(See inline comments for GC diagnosis and remediation guidance.)")

## Section 9 — Spill detection (comment + config)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 9: Spill Detection in Stages Tab")
print("=" * 70)

# Spill to disk: shuffle data exceeds spark.shuffle.spill threshold.
# Signs in UI:   Stages tab shows "Spill (Disk)" column with non-zero values.
# Common causes: shuffle.partitions too low → huge per-partition size.
# Fix:           increase shuffle.partitions OR repartition before shuffle-heavy op.
print("\nSpill occurs when executor memory is insufficient for shuffle buffers.")
print("Watch 'Spill (Disk)' column in Stages tab > click any stage > scroll to Task Metrics.")

## Cleanup and summary

In [ ]:
emp_cached.unpersist()

print("\nAll UI demos complete.")
print("Tabs to review before stopping:")
print("  Jobs:        each action = one row")
print("  Stages:      shuffle stages show Exchange; skipped = cached")
print("  SQL:         DataFrame operations with query plans")
print("  Storage:     cached DataFrames with memory usage")
print("  Environment: all config values")

stop_and_wait(spark)